In [1]:
import sys
sys.path.append("../src/")
%load_ext autoreload
%autoreload 2

In [2]:
from ipm_new.ipm_solve import ipm_solve # type: ignore[reportMissingImports]
from relaxers.remap_lower_bounds import remap_lower_bounds # type: ignore[reportMissingImports]
from relaxers.quadratic_model_lin_approx import quadratic_model_lin_approx # type: ignore[reportMissingImports]
from relaxers.quadratic_lin_approx_no_surrogate import quadratic_lin_approx_no_surrogate # type: ignore[reportMissingImports]
from relaxers.quadratic_lin_approx_nosur_refine import quadratic_lin_approx_no_surrogate_refine # type: ignore[reportMissingImports]
from testers.classical_solve_lin_program import solve_lp_return_x # type: ignore[reportMissingImports]
from testers.classical_solve_quad_program import solve_qcp_return_x # type: ignore[reportMissingImports]
from testers.condition_number_lin_program import condition_number_nes_basic # type: ignore[reportMissingImports]

import matplotlib.pyplot as plt
import numpy as np

In [3]:
LIN_APPROX = 10

# Whether to do an inner or outer approximation
OUTER_APPROXIMATION = True

# Whether the coefficient is part of the surrogate
COEFFICIENT_SURROGATE = True

# Whether to bound the surrogates with bound constraints.
SURROGATE_BOUND_BELOW = True
SURROGATE_BOUND_ABOVE = True

# Whether to divide by lambda in an inner approximation
REMOVE_DIVISION = False

In [4]:
quad, q_time, q_x = solve_qcp_return_x(f_name="model11_quad_reform_nofix.json")
print(quad)

Set parameter Username
Set parameter LicenseID to value 2823041
Academic license - for non-commercial use only - expires 2027-05-16
2914.0372553299326


In [ ]:
quadratic_lin_approx_no_surrogate(LIN_APPROX, OUTER_APPROXIMATION, remove_division=REMOVE_DIVISION,
                            f_name="model11_quad_reform_nofix.json", endpoints=False)
remap_lower_bounds(f_name="linear_approx.json")
val, time, x, compl = ipm_solve()
print(x)

print(condition_number_nes_basic())
print(f"Error: {val - quad}")

19800.0
Condition number before pre-conditioning: 4331.471021194637
Condition number after pre-conditioning: 28.388998993352747
Iteration 1:
Primal objective:   4.43662819e+09 
Dual objective:     -2.56150765e+04

Primal residual:    4.43e+06
Dual residual:      1.57e+06
Complementarity:    8.92e+09

Condition number before pre-conditioning: 1277.6322042427112
Condition number after pre-conditioning: 16.30814190023224
Iteration 2:
Primal objective:   3.87926566e+09 
Dual objective:     -2.55575229e+04

Primal residual:    3.89e+06
Dual residual:      1.38e+06
Complementarity:    7.93e+09

Condition number before pre-conditioning: 934.8615578593123
Condition number after pre-conditioning: 14.937854400867002
Iteration 3:
Primal objective:   2.97472284e+09 
Dual objective:     -2.42835500e+04

Primal residual:    3.00e+06
Dual residual:      1.06e+06
Complementarity:    6.25e+09

Condition number before pre-conditioning: 531.6957366916532
Condition number after pre-conditioning: 13.801425

In [6]:
# Define a set of functions for each

quads = [[val] for val in x[:10]]
# print(quads)
iters = 5

for i in range(1, iters + 1):
    print(f"Iteration {i}")
    points_functions = []

    def np_uniform_add_q_factory(qs):
        def np_uniform_add_qs(lower, upper, num):
                refinement_points = len(qs)
                uniform = np.linspace(lower, upper, num - refinement_points)
                return np.append(uniform, qs)
        return np_uniform_add_qs

    for qs in quads:
        points_functions.append(np_uniform_add_q_factory(qs))

    quadratic_lin_approx_no_surrogate_refine(LIN_APPROX + i, OUTER_APPROXIMATION, points_function=points_functions,
                                                remove_division=REMOVE_DIVISION, f_name="model11_quad_reform_nofix.json",
                                                endpoints=False)
    remap_lower_bounds(f_name="linear_approx.json")

    val, time, x, compl = ipm_solve(f_name="linear_approx.json")

    # print(f"x: {x}")
    print(f"val: {val}")

    for j, qs in enumerate(quads):
        qs.append(x[j])

    # print(quads)
    print(compl)

print(x)
print(f"Error: {val - quad}")

Iteration 1
The solution quality is limited by the precision of the linear system solver.
The algorithm stopped after 96 iterations in 7.12 seconds.

Primal objective:   -2.58377318e+03
Dual objective:     -2.65329007e+03

Primal residual:    2.39e-03
Dual residual:      8.01e-04
Complementarity:    7.45e+01

val: 2653.2900713346735
74.52562200290781
Iteration 2
The solution quality is limited by the precision of the linear system solver.
The algorithm stopped after 124 iterations in 9.18 seconds.

Primal objective:   -2.87980934e+03
Dual objective:     -2.91472173e+03

Primal residual:    1.16e-03
Dual residual:      3.84e-04
Complementarity:    3.77e+01

val: 2914.7217342513286
37.69449035973389
Iteration 3
The solution quality is limited by the precision of the linear system solver.
The algorithm stopped after 125 iterations in 9.37 seconds.

Primal objective:   -2.81016279e+03
Dual objective:     -2.92162146e+03

Primal residual:    4.04e-03
Dual residual:      1.31e-03
Complementa